Приёмник под сигнал с преамбулой Голея и паузами (`test_transmit.ipynb`).

Параметры `n_bits`, `samples_per_chip`, `gap_seconds`, `num_bursts` и `sample_rate` / `center_freq` должны совпадать с передатчиком.

In [ ]:
import sys
sys.path.append('/home/pavel/pyadi-iio/')
sys.path.append('/home/pavel/libiio')
import adi
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def golay_complementary_pair(n_bits=4):
    """Та же рекурсия, что на передаче."""
    A = np.array([1, 1], dtype=np.int8)
    B = np.array([1, -1], dtype=np.int8)
    for _ in range(n_bits - 1):
        A, B = np.concatenate([A, B]), np.concatenate([A, -B])
    return A, B


def preamble_template(n_bits, samples_per_chip):
    """Ожидаемая преамбула (без паузы), нормированная под согласованный фильтр."""
    A, _ = golay_complementary_pair(n_bits)
    chips = A.astype(np.float64)
    ref = np.repeat(chips, samples_per_chip)
    ref = ref / (np.linalg.norm(ref) + 1e-12)
    return ref.astype(np.complex128)


def matched_filter_preamble(rx, ref):
    """
    Согласованный фильтр: корреляция с эталоном преамбулы.
    Пик, когда преамбула совпала с окном; индекс пика в `mf` ≈ (начало преамбулы в rx) + len(ref) - 1
    при выходе `np.convolve(..., mode='full')`.
    """
    rx = np.asarray(rx).ravel()
    ref = np.asarray(ref).ravel()
    mf = np.convolve(rx, np.conj(ref[::-1]), mode="full")
    return mf


def detect_preamble_starts(mf, ref_len, min_height_ratio=0.25):
    """
    Грубые оценки положений начал преамбул по пикам |mf|.
    Индекс начала преамбулы в rx ≈ peak_idx - (ref_len - 1).
    """
    mag = np.abs(mf)
    thr = np.max(mag) * min_height_ratio
    peaks = []
    for i in range(1, len(mag) - 1):
        if mag[i] >= mag[i - 1] and mag[i] >= mag[i + 1] and mag[i] >= thr:
            peaks.append(i)
    if not peaks:
        return np.array([], dtype=int), mag
    peaks = np.array(peaks)
    starts = peaks - (ref_len - 1)
    return starts, mag

In [ ]:
# Должно совпадать с test_transmit.ipynb
sample_rate = 20e6
center_freq = 2.71e9
n_bits = 4
samples_per_chip = 8
gap_seconds = 100e-6
num_bursts = 4

A, _ = golay_complementary_pair(n_bits)
n_preamble_samp = len(A) * samples_per_chip
gap_samples = max(0, int(round(gap_seconds * sample_rate)))
burst_period = n_preamble_samp + gap_samples

ref = preamble_template(n_bits, samples_per_chip)
ref_len = len(ref)
print(
    f"Эталон: {ref_len} отсч, период burst+gap: {burst_period} отсч (~{burst_period/sample_rate*1e6:.2f} µs)"
)

In [ ]:
sdr = adi.ad9363("ip:192.168.2.1")

sdr.rx_enabled_channels = [0, 1]
sdr.sample_rate = int(sample_rate)
sdr.rx_rf_bandwidth = int(sample_rate)
sdr.rx_lo = int(center_freq)
# не меньше одного полного периода буфера передатчика (см. print на tx)
sdr.rx_buffer_size = max(16384, burst_period * num_bursts + ref_len)
sdr.gain_control_mode_chan0 = "manual"
sdr.gain_control_mode_chan1 = "manual"
sdr.rx_hardwaregain_chan0 = int(30)
sdr.rx_hardwaregain_chan1 = int(30)

In [ ]:
rx = sdr.rx()
rx0 = rx[0]

mf = matched_filter_preamble(rx0, ref)
starts, mag = detect_preamble_starts(mf, ref_len)

peak_idx = int(np.argmax(np.abs(mf)))
est_start = peak_idx - (ref_len - 1)
print(f"Макс. |корреляция| на выходе MF: индекс {peak_idx}, оценка начала преамбулы в rx: {est_start}")
if len(starts):
    print(f"Найдено пиков (грубо): {len(starts)}, первые старты: {starts[:8]}")

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=False)
n_plot = min(len(rx0), 3 * burst_period)
ax[0].plot(np.real(rx0[:n_plot]), label="Re", alpha=0.9)
ax[0].plot(np.imag(rx0[:n_plot]), label="Im", alpha=0.7)
ax[0].set_title("Принятый сигнал (фрагмент)")
ax[0].set_xlabel("отсчёт")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

lag = np.arange(len(mf)) - (ref_len - 1)
ax[1].plot(lag[: len(mf)], np.abs(mf))
ax[1].axvline(est_start, color="r", linestyle="--", alpha=0.7, label="оценка старта")
ax[1].set_title("|Согласованный фильтр| (ось — лаг, 0 ≈ начало преамбулы в rx)")
ax[1].set_xlabel("лаг (отсчёты)")
ax[1].legend()
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()